# 05 · Modellvergleich & Analyse — Favorita

Vollständiger Vergleich aller Modelle:
- Metriken auf Val- und Test-Set
- Feature Importance (XGBoost & LightGBM)
- Tiefergehende Modellanalyse
- Fehleranalyse pro Store
- Residuenanalyse des besten Modells

## 0 · Imports & Setup

In [2]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import time
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
torch.set_num_threads(1)
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST, NHITS

sys.path.append(os.path.abspath('../03_src'))
from config import FINAL, TARGET_COL, FEATURE_COLS, EXOG_COLS, STAT_EXOG, HIST_EXOG, TRAIN_END, VAL_END, LOOKBACK, HORIZON, MODELL_ORDER, MODELL_COLORS, PRED_COLS, RESULTS, FILE_NAMES, LGBM_PARAMS, XGB_PARAMS, PATCHTST_PARAMS, NHITS_PARAMS
from utilis import run_sarimax, run_prophet, run_xgb, run_lgbm, to_nf_format, compute_metrics, load_preds, nf_to_polars

sns.set_style('whitegrid')

print('Setup ✓')

Setup ✓


## 1 · Daten laden

In [3]:
df = pl.read_parquet(FINAL / 'final_dataset.parquet')

train     = df.filter(pl.col('date') <= TRAIN_END)
val       = df.filter((pl.col('date') > TRAIN_END) & (pl.col('date') <= VAL_END))
test      = df.filter(pl.col('date') > VAL_END)
train_val = df.filter(pl.col('date') <= VAL_END)

stores = sorted(df['store_nbr'].unique().to_list())

print(f'Train:     {train.shape}  {train["date"].min()} → {train["date"].max()}')
print(f'Val:       {val.shape}    {val["date"].min()} → {val["date"].max()}')
print(f'Test:      {test.shape}   {test["date"].min()} → {test["date"].max()}')
print(f'Stores:    {len(stores)}')

Train:     (70114, 37)  2013-01-29 → 2016-12-31
Val:       (7965, 37)    2017-01-01 → 2017-05-31
Test:      (4104, 37)   2017-06-01 → 2017-08-15
Stores:    54


In [4]:
# Val-Vorhersagen laden
val_preds = {m: load_preds(f'val_{m.lower()}.parquet') for m in MODELL_ORDER}
available = [m for m, df_m in val_preds.items() if df_m is not None]
print(f'Verfügbare Val-Modelle: {available}')

Verfügbare Val-Modelle: ['SARIMAX', 'Prophet', 'XGBoost', 'LightGBM', 'PatchTST', 'NHITS']


In [5]:
# TEstzeiten wegspeichern
test_times = {}

## 2 · Val-Set Metriken

In [6]:
metrics_val = []

for modell in available:
    preds_df = val_preds[modell]
    pred_col = PRED_COLS[modell]
    merged   = preds_df.drop_nulls(subset=[pred_col, 'y_true'])
    metrics_val.append(
        compute_metrics(merged['y_true'].to_numpy(), merged[pred_col].to_numpy(), modell, 'val')
    )

metrics_val_df = pd.DataFrame(metrics_val).set_index('modell').drop(columns='split')
print('=== Val-Set Metriken ===')
print(metrics_val_df.sort_values('MAE').round(2))

=== Val-Set Metriken ===
             MAE     RMSE   MAPE
modell                          
XGBoost     8.62    24.67   0.57
LightGBM   12.45    25.92   0.82
Prophet   123.95   194.00   8.58
NHITS     269.92   365.72  19.25
PatchTST  331.21   423.81  25.72
SARIMAX   816.30  1671.33  66.04


## 3 · Test-Set: Re-Run aller Modelle

# SARIMAX

In [7]:
t0 = time.time()

test_sarimax = run_sarimax(train_val, test, stores=stores, exog_cols=EXOG_COLS)
test_times['SARIMAX'] = round(time.time() - t0, 1)
test_sarimax.write_parquet(RESULTS / 'test_sarimax.parquet')
print(f"SARIMAX ✓  {test_times['SARIMAX']}s")

SARIMAX:   0%|          | 0/54 [00:00<?, ?it/s]c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No sup

SARIMAX ✓  293.4s


# Prophet

In [8]:
t0 = time.time()


test_prophet = run_prophet(train_val, test, stores=stores, extra_regressors=['oil_price'])
test_times['Prophet'] = round(time.time() - t0, 1)
test_prophet.write_parquet(RESULTS / 'test_prophet.parquet')
print('Prophet ✓')

Prophet Progress:   0%|          | 0/54 [00:00<?, ?it/s]21:50:31 - cmdstanpy - INFO - Chain [1] start processing
21:50:31 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   2%|▏         | 1/54 [00:00<00:42,  1.26it/s]21:50:31 - cmdstanpy - INFO - Chain [1] start processing
21:50:32 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   4%|▎         | 2/54 [00:01<00:32,  1.62it/s]21:50:32 - cmdstanpy - INFO - Chain [1] start processing
21:50:32 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   6%|▌         | 3/54 [00:01<00:25,  1.97it/s]21:50:32 - cmdstanpy - INFO - Chain [1] start processing
21:50:32 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   7%|▋         | 4/54 [00:02<00:22,  2.18it/s]21:50:33 - cmdstanpy - INFO - Chain [1] start processing
21:50:33 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   9%|▉         | 5/54 [00:02<00:20,  2.36it/s]21:50:33 - cmdstanpy - INFO - Chain [1] start processing
21

Prophet ✓


# XGBoost & LightGBM

In [9]:
X_train_val = train_val.select(FEATURE_COLS).to_numpy()
y_train_val = train_val.select(TARGET_COL).to_numpy().ravel()
X_test      = test.select(FEATURE_COLS).to_numpy()
y_test      = test.select(TARGET_COL).to_numpy().ravel()

t_xgb = time.time()
model_xgb, _ = run_xgb(X_train_val, y_train_val, X_test, y_test, **XGB_PARAMS)
test_times['XGBoost'] = round(time.time() - t_xgb, 1)

t_lgbm = time.time()
model_lgbm, _ = run_lgbm(X_train_val, y_train_val, X_test, y_test, **LGBM_PARAMS)
test_times['LightGBM'] = round(time.time() - t_lgbm, 1)

test_xgb = (
    test.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'})
    .with_columns(pl.Series('pred_xgb', model_xgb.predict(X_test).astype(float)))
)

test_lgbm = (
    test.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'})
    .with_columns(pl.Series('pred_lgbm', model_lgbm.predict(X_test).astype(float)))
)

# 3. Parquet-Exporte und Modell-Sicherung
test_xgb.write_parquet(RESULTS / 'test_xgboost.parquet')
test_lgbm.write_parquet(RESULTS / 'test_lightgbm.parquet')

joblib.dump(model_xgb, RESULTS / 'model_xgb_final.pkl')
joblib.dump(model_lgbm, RESULTS / 'model_lgbm_final.pkl')

print('XGBoost & LightGBM ✓')

XGBoost Val           MAE=6.3  RMSE=12.2  MAPE=0.4%
LightGBM Val          MAE=9.5  RMSE=14.7  MAPE=0.6%
XGBoost & LightGBM ✓


# PatchTST & NHITS

In [10]:
min_required_len = LOOKBACK + HORIZON


train_val_clean = (
    train_val
    .filter(pl.len().over("store_nbr") >= min_required_len)
)

ursprung_stores = train_val["store_nbr"].n_unique()
bereinigte_stores = train_val_clean["store_nbr"].n_unique()
if ursprung_stores != bereinigte_stores:
    print(f"⚠️ {ursprung_stores - bereinigte_stores} Store(s) wurden entfernt, da sie weniger als {min_required_len} Datenpunkte hatten.")

train_val_nf = to_nf_format(train_val_clean)

⚠️ 1 Store(s) wurden entfernt, da sie weniger als 412 Datenpunkte hatten.


In [11]:
static_df = (
    train_val_clean.select(['store_nbr'] + STAT_EXOG)
                   .unique(subset=['store_nbr']).sort('store_nbr')
                   .with_columns(pl.col('store_nbr').cast(pl.Utf8).alias('unique_id'))
                   .select(['unique_id'] + STAT_EXOG)
                   .with_columns([pl.col(c).cast(pl.Float32) for c in STAT_EXOG])
                   .to_pandas()
)

t_pt = time.time()

model_patchtst_final = PatchTST(
    h=HORIZON, 
    input_size=LOOKBACK,
    max_steps=200, 
    early_stop_patience_steps=10, 
    val_check_steps=25, 
    **PATCHTST_PARAMS
)
nf_pt = NeuralForecast(models=[model_patchtst_final], freq='D')
nf_pt.fit(df=train_val_nf[['unique_id', 'ds', 'y']], val_size=HORIZON)

test_times['PatchTST'] = round(time.time() - t_pt, 1)

test_patchtst = nf_to_polars(nf_pt.predict(), 'PatchTST', 'pred_patchtst', test)
test_patchtst.write_parquet(RESULTS / 'test_patchtst.parquet')
print('PatchTST ✓')

Seed set to 42


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name         | Type              | Params | Mode  | FLOPs
-------------------------------------------------------------------
0 | loss         | MAE               | 0      | train | 0    
1 | padder_train | ConstantPad1d     | 0      | train | 0    
2 | scaler       | TemporalNorm      | 0      | train | 0    
3 | model        | PatchTST_backbone | 472 K  | train | 0    
-------------------------------------------------------------------
472 K     Trainable params
2         Non-trainable params
472 K     Total params
1.888     Total estimated model params size (MB)
65        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=200` reached.
Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

PatchTST ✓


In [12]:
# NHITS

required_cols_nhits = ['unique_id', 'ds', 'y'] + HIST_EXOG

t_nh = time.time()

model_nhits_final = NHITS(
    h=HORIZON,
    input_size=LOOKBACK,
    stat_exog_list=STAT_EXOG,
    hist_exog_list=HIST_EXOG,
    max_steps=200,  # Volle Steps für das finale Ergebnis
    **NHITS_PARAMS
)

if 'pooling_widths' in model_nhits_final.trainer_kwargs:
    del model_nhits_final.trainer_kwargs['pooling_widths']

nf_nh = NeuralForecast(models=[model_nhits_final], freq='D')

# NHITS trainiert auf dem vollen DataFrame inkl. Features und nutzt static_df
nf_nh.fit(
    df=train_val_nf[required_cols_nhits], 
    static_df=static_df, 
    val_size=HORIZON
)

test_times['NHITS'] = round(time.time() - t_nh, 1)

test_nhits = nf_to_polars(nf_nh.predict(), 'NHITS', 'pred_nhits', test)
test_nhits.write_parquet(RESULTS / 'test_nhits.parquet')
print('NHITS ✓')

Seed set to 42
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\neuralforecast\tsdataset.py:129: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  x = torch.from_numpy(x)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name         | Type          | Params | Mode  | FLOPs
-----------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

NHITS ✓


In [13]:
print('\n── Trainingszeiten (Sekunden) ──')
for m, t in test_times.items():
    print(f'  {m:<12} {t:>8.1f}s')

times_df = pl.DataFrame({
    "Modell": list(test_times.keys()),
    "Zeit_Sekunden": list(test_times.values())
})
times_df.write_parquet(RESULTS / 'testszeit.parquet')


── Trainingszeiten (Sekunden) ──
  SARIMAX         293.4s
  Prophet          20.9s
  XGBoost           9.2s
  LightGBM          2.3s
  PatchTST       1380.4s
  NHITS            81.4s
